In [81]:
import numpy as np
import pickle
import casadi as ca
import time

num_var = 100
num_ineq = 50
num_eq = 50
num_examples = 10
seed = 2025

print("Nonsmooth nonconvex SOCP problem with {} variables, {} inequalities, {} equalities and {} examples".format(num_var, num_ineq, num_eq, num_examples))
np.random.seed(seed)
Q = np.diag(np.random.rand(num_var)*0.5)
p = np.random.uniform(-1, 1, num_var)
A = np.random.uniform(-1, 1, size=(num_eq, num_var))
X = np.random.uniform(-1, 1, size=(num_examples, num_eq))
XL = X.min(axis=0)
XU = X.max(axis=0)

L = np.ones((num_var))*-5
U = np.ones((num_var))*5
x0 = np.random.uniform(-1, 1, size=(num_var))
G = []
h = []
C = []
d = []
for i in range(num_ineq):
    G.append(np.random.uniform(-1, 1, size=(num_ineq, num_var)))
    h.append(np.random.uniform(-1, 1, size=(num_ineq)))
    C.append(np.random.uniform(-1, 1, size=(num_var)))
    d.append(np.linalg.norm(G[i] @ x0 + h[i], 2) - C[i].T @ x0)
data = {'Q':Q,
        'p':p,
        'A':A,
        'X':X,
        'G':np.array(G),
        'h':np.array(h),
        'C':np.array(C),
        'd':np.array(d),
        'YL':L,
        'YU':U,
        'XL':XL,
        'XU':XU,
        'Y':[],
        'instance_ipopt_times':[],
        'instance_py_times':[]}

Nonsmooth nonconvex SOCP problem with 100 variables, 50 inequalities, 50 equalities and 10 examples


In [82]:
Y = []
for n in range(num_examples):
    # print("Solving example {}".format(n))
    Xi = X[n]
    y = ca.MX.sym('y_var', num_var)
    t = ca.MX.sym('t_var')

    obj_func = 0.5 * ca.mtimes(y.T, ca.mtimes(Q, y)) + ca.dot(p, ca.sin(y)) + 0.1*t

    eq_constraints = A @ y - Xi
    soc = ca.dot(y, y) - t**2
    ineq_constraints = []
    for i in range(num_ineq):
        ineq_constraints.append(ca.norm_2(G[i] @ ca.cos(y) + h[i]) - (ca.dot(C[i], y) + d[i]))
    ineq_constraints.append(soc)
    ineq_constraints = ca.vertcat(*ineq_constraints)
    
    nlp = {'x': ca.vertcat(y, t), 'f': obj_func, 'g': ca.vertcat(eq_constraints, ineq_constraints)}
    # opts = {'ipopt.print_level': 0, 'print_time': 0, }
    budgets = [2, 30, 100, 300]
    max_iter = 2 # 10 30 90 150 300 
    max_cpu_time = 10.0

    opts = {
        "ipopt.print_level": 0,
        "print_time": 0,

        # Early exit
        # "ipopt.acceptable_iter": 1,

        # # Loose acceptable criteria
        # "ipopt.acceptable_tol": 1e-3,
        # "ipopt.acceptable_constr_viol_tol": 1e-3,
        # "ipopt.acceptable_dual_inf_tol": 1e-3,
        # "ipopt.acceptable_compl_inf_tol": 1e-3,
    }

    if max_iter is not None:
        # opts["ipopt.max_iter"] = max_iter
        opts["ipopt.max_cpu_time"] = max_cpu_time  # seconds

    solver = ca.nlpsol('solver', 'ipopt', nlp, opts)
    # Define bounds for variables and constraints
    lbg = np.concatenate([np.zeros(num_eq), -np.inf * np.ones(num_ineq+1)])
    ubg = np.concatenate([np.zeros(num_eq), np.zeros(num_ineq+1)])
    lbx = np.concatenate([L, [0]])
    ubx = np.concatenate([U, [np.inf]])

    start_time = time.time()
    res = solver(lbg=lbg, ubg=ubg, lbx=lbx, ubx=ubx)
    python_wall = time.time() - start_time

    stats = solver.stats()

    solver_wall = sum(
        stats[k] for k in stats if k.startswith("t_wall_")
    )

    print(f"Python wall time : {python_wall:.4f} s")
    print(f"IPOPT wall time  : {solver_wall:.4f} s")
    data['instance_ipopt_times'].append(solver_wall)
    data['instance_py_times'].append(python_wall)

    # check if the solver converged
    # if solver.stats()['success']:
    #     sol_x = res['x'].full().flatten()
    #     Y.append(sol_x[:-1])
    # else:
    #     print("Solver failed to converge")
    #     break

    sol_x = res['x'].full().flatten()
    Y.append(sol_x[:-1])

    success = solver.stats()['success']
    print("Example {} - Success: {}: Objective value: {}".format(n, success, res['f'].full().flatten()[0]))

data['Y'] = np.array(Y)


i = 0
det_min = 0
best_partial = 0
while i < 1000:
    np.random.seed(i)
    partial_vars = np.random.choice(num_var, num_var - num_eq, replace=False)
    other_vars = np.setdiff1d(np.arange(num_var), partial_vars)
    _, det = np.linalg.slogdet(A[:, other_vars])
    if det>det_min:
        det_min = det
        best_partial = partial_vars
    i += 1
print('best_det', det_min)
data['best_partial'] = best_partial


# with open("datasets/nonsmooth_nonconvex/socp/random{}_socp_dataset_var{}_ineq{}_eq{}_ex{}".format(seed, num_var, num_ineq, num_eq, num_examples), 'wb') as f:
#     pickle.dump(data, f)

Python wall time : 6.5192 s
IPOPT wall time  : 5.9823 s
Example 0 - Success: True: Objective value: -5.8670490089395955
Python wall time : 5.3130 s
IPOPT wall time  : 4.8566 s
Example 1 - Success: True: Objective value: -8.390675691290351
Python wall time : 4.1024 s
IPOPT wall time  : 3.7906 s
Example 2 - Success: True: Objective value: -7.4383153208483375
Python wall time : 5.5933 s
IPOPT wall time  : 5.2080 s
Example 3 - Success: True: Objective value: -6.8693772238084945
Python wall time : 3.5436 s
IPOPT wall time  : 3.2565 s
Example 4 - Success: True: Objective value: -8.236584289781758
Python wall time : 3.5896 s
IPOPT wall time  : 3.3205 s
Example 5 - Success: True: Objective value: -6.8693063485463615
Python wall time : 3.3338 s
IPOPT wall time  : 3.1109 s
Example 6 - Success: True: Objective value: -9.411724078038203
Python wall time : 6.8535 s
IPOPT wall time  : 6.2783 s
Example 7 - Success: False: Objective value: -7.966365542085432
Python wall time : 4.8723 s
IPOPT wall time

In [83]:
# Python wall time : 4.1379 s
# IPOPT wall time  : 3.8211 s
# Example 0: Objective value: 21.830696173485194
# Python wall time : 5.1073 s
# IPOPT wall time  : 4.7339 s
# Example 1: Objective value: 20.49899542514615
# best_det 50.35299906971193

In [84]:
# Python wall time : 5.9450 s
# IPOPT wall time  : 5.5313 s
# Example 0: Objective value: 9.150703123482039
# Python wall time : 7.8945 s
# IPOPT wall time  : 7.3162 s
# Example 1: Objective value: -0.2152281530008182
# best_det 50.35299906971193

In [85]:
# # print all input and output data
# for key, value in data.items():
#     print(key, value)

In [86]:
# import numpy as np
# from scipy.optimize import minimize, NonlinearConstraint, Bounds


# # -----------------------------
# # Objective
# # -----------------------------
# def objective(z):
#     y = z[:-1]
#     t = z[-1]
#     return 0.5 * y @ Q @ y + p @ np.sin(y) + 0.1 * t

# # -----------------------------
# # Constraints
# # -----------------------------
# def equality_constraint(z, x):
#     y = z[:-1]
#     return A @ y - x

# def soc_constraint(z):
#     y = z[:-1]
#     t = z[-1]
#     return np.dot(y, y) - t**2

# def nonsmooth_constraints(z):
#     y = z[:-1]
#     return np.array([
#         np.linalg.norm(G[i] @ np.cos(y) + h[i]) - (C[i] @ y + d[i])
#         for i in range(num_ineq)
#     ])


# # -----------------------------
# # Solve for each X
# # -----------------------------
# Y2 = []

# for n in range(num_examples):
#     x = X[n]

#     eq_con = NonlinearConstraint(
#         lambda z, x=x: equality_constraint(z, x),
#         lb=np.zeros(num_eq),
#         ub=np.zeros(num_eq)
#     )

#     soc_con = NonlinearConstraint(
#         soc_constraint,
#         lb=-np.inf,
#         ub=0.0
#     )

#     nonsmooth_con = NonlinearConstraint(
#         nonsmooth_constraints,
#         lb=-np.inf * np.ones(num_ineq),
#         ub=np.zeros(num_ineq)
#     )

#     bounds = Bounds(
#         np.concatenate([L, [0.0]]),
#         np.concatenate([U, [np.inf]])
#     )

#     z0 = np.concatenate([np.random.uniform(L, U), [1.0]])

#     res = minimize(
#         objective,
#         z0,
#         method="SLSQP",
#         bounds=bounds,
#         constraints=[eq_con, soc_con, nonsmooth_con],
#         options={"ftol": 1e-6}
#     )

#     if not res.success:
#         print(f"Failed at sample {n}: {res.message}")
#         break

#     Y2.append(res.x[:-1])

#     print(f"Sample {n}, obj = {res.fun:.6f}")

# Y2 = np.array(Y2)
# print("Finished SciPy optimization")


In [87]:
# # print solutions
# print("Y:", Y)
# print("Y2:", Y2)
# print("Difference between Y and Y2:", np.linalg.norm(Y - Y2))

In [88]:
def constraint_violation(y, t, x):
    # Equality violation
    eq_violation = np.linalg.norm(A @ y - x, ord=2)

    # Inequality violations
    ineq_vals = []
    for i in range(num_ineq):
        gi = np.linalg.norm(G[i] @ np.cos(y) + h[i]) - (C[i] @ y + d[i])
        ineq_vals.append(max(0.0, gi))

    soc = np.dot(y, y) - t**2
    ineq_vals.append(max(0.0, soc))

    ineq_vals = np.array(ineq_vals)

    # Bound violations
    lb_violation = np.maximum(0.0, L - y)
    ub_violation = np.maximum(0.0, y - U)
    t_violation  = max(0.0, -t)

    bound_violation = np.linalg.norm(
        np.concatenate([lb_violation, ub_violation, [t_violation]]),
        ord=2
    )

    return {
        "eq_l2": eq_violation,
        "ineq_max": ineq_vals.max(),
        "ineq_l2": np.linalg.norm(ineq_vals, ord=2),
        "bound_l2": bound_violation
    }
def grad_objective(y, t):
    grad_y = Q @ y + p * np.cos(y)
    grad_t = np.array([0.1])
    return np.concatenate([grad_y, grad_t])
def jacobian_eq():
    J = np.zeros((num_eq, num_var + 1))
    J[:, :num_var] = A
    return J

def jacobian_soc(y, t):
    J = np.zeros(num_var + 1)
    J[:num_var] = 2 * y
    J[-1] = -2 * t
    return J

def jacobian_nonsmooth(y, i):
    v = G[i] @ np.cos(y) + h[i]
    norm_v = np.linalg.norm(v)

    if norm_v < 1e-8:
        return np.zeros(num_var + 1)

    J = np.zeros(num_var + 1)
    J[:num_var] = (
        -(G[i].T @ (v / norm_v)) * np.sin(y) - C[i]
    )
    return J
def kkt_residual(y, t, x):
    grad_f = grad_objective(y, t)

    # Active constraints
    J = []
    rhs = -grad_f

    # Equality constraints
    J.append(jacobian_eq())

    # Active inequalities
    for i in range(num_ineq):
        gi = np.linalg.norm(G[i] @ np.cos(y) + h[i]) - (C[i] @ y + d[i])
        if gi > -1e-6:
            J.append(jacobian_nonsmooth(y, i)[None, :])

    soc = np.dot(y, y) - t**2
    if soc > -1e-6:
        J.append(jacobian_soc(y, t)[None, :])

    if not J:
        return np.linalg.norm(grad_f)

    J = np.vstack(J)

    # Least-squares multipliers
    try:
        lam, *_ = np.linalg.lstsq(J.T, rhs, rcond=None)
        res = grad_f + J.T @ lam
        return np.linalg.norm(res)
    except np.linalg.LinAlgError:
        return np.inf

In [89]:
# y = Y2[-1]      # Scipy solution
# t = res.x[-1]
# x = X[-1]

# y = Y[-1]      # IPOPT solution
# t = sol_x[-1]
# x = X[-1]


# viol = constraint_violation(y, t, x)
# kkt  = kkt_residual(y, t, x)

# print("Constraint violation:", viol)
# print("KKT residual:", kkt)

# compute and print average constraint violation and KKT residual over all examples
total_viol = {"eq_l2": 0.0,
              "ineq_max": 0.0,
              "ineq_l2": 0.0,
              "bound_l2": 0.0}
total_kkt = 0.0
ipopt_times = 0.0
py_times = 0.0
for n in range(num_examples):
    y = Y[n]
    t = sol_x[-1]
    x = X[n]
    ipopt_time = data['instance_ipopt_times'][n]
    py_time = data['instance_py_times'][n]

    viol = constraint_violation(y, t, x)
    kkt  = kkt_residual(y, t, x)

    for key in total_viol:
        total_viol[key] += viol[key]
    total_kkt += kkt
    ipopt_times += ipopt_time
    py_times += py_time
for key in total_viol:
    total_viol[key] /= num_examples
total_kkt /= num_examples
ipopt_times /= num_examples
py_times /= num_examples
print("Average IPOPT time:", ipopt_times)
print("Average Python time:", py_times)
print("Average constraint violation:", total_viol)
print("Average KKT residual:", total_kkt)

Average IPOPT time: 4.4845275313
Average Python time: 4.863925409317017
Average constraint violation: {'eq_l2': np.float64(1.339324312090719e-14), 'ineq_max': np.float64(5.6755556785095225), 'ineq_l2': np.float64(5.675555678883945), 'bound_l2': np.float64(0.0)}
Average KKT residual: 0.03502393076040319


2 OK
Average IPOPT time: 0.08656705719999999
Average Python time: 0.09722776412963867
Average constraint violation: {'eq_l2': np.float64(4.01497815321685), 'ineq_max': np.float64(28.785072326690948), 'ineq_l2': np.float64(123.35926525148167), 'bound_l2': np.float64(0.0)}
Average KKT residual: 1.8095087397219318e-13

10
Average IPOPT time: 0.40617983080000003
Average Python time: 0.4457808494567871
Average constraint violation: {'eq_l2': np.float64(3.927604037582614), 'ineq_max': np.float64(28.030853786134465), 'ineq_l2': np.float64(119.37388035095509), 'bound_l2': np.float64(0.0)}
Average KKT residual: 3.691799800409053e-13

30 OK
Average IPOPT time: 1.2051251048
Average Python time: 1.312480592727661
Average constraint violation: {'eq_l2': np.float64(1.8412055065064024), 'ineq_max': np.float64(70.72903691025658), 'ineq_l2': np.float64(107.13145339653697), 'bound_l2': np.float64(0.0)}
Average KKT residual: 0.7411526968533979

60 
Average IPOPT time: 2.3244319338999992
Average Python time: 2.529162549972534
Average constraint violation: {'eq_l2': np.float64(0.4503438485639079), 'ineq_max': np.float64(28.566233281179045), 'ineq_l2': np.float64(37.885249171143684), 'bound_l2': np.float64(0.0)}
Average KKT residual: 1.698714234669803

90
Average IPOPT time: 2.9953926177
Average Python time: 3.265803074836731
Average constraint violation: {'eq_l2': np.float64(0.032317404396934785), 'ineq_max': np.float64(8.275643142358575), 'ineq_l2': np.float64(8.393518216030568), 'bound_l2': np.float64(0.0)}
Average KKT residual: 2.555077545835896

100
Average IPOPT time: 3.7147936064
Average Python time: 4.069062447547912
Average constraint violation: {'eq_l2': np.float64(0.023221386440856237), 'ineq_max': np.float64(14.329080350469017), 'ineq_l2': np.float64(14.374427748838537), 'bound_l2': np.float64(0.0)}
Average KKT residual: 1.4243637406425247

120
Average IPOPT time: 3.1662965081999994
Average Python time: 3.461210584640503
Average constraint violation: {'eq_l2': np.float64(1.3314770357501998e-14), 'ineq_max': np.float64(11.362956304694311), 'ineq_l2': np.float64(11.362956304694311), 'bound_l2': np.float64(0.0)}
Average KKT residual: 2.779391459117081

150
Average IPOPT time: 3.5631077602999994
Average Python time: 3.834502911567688
Average constraint violation: {'eq_l2': np.float64(1.5060324967894074e-14), 'ineq_max': np.float64(5.864477971478057), 'ineq_l2': np.float64(5.864480903780363), 'bound_l2': np.float64(0.0)}
Average KKT residual: 0.4927374219993198

300
Average IPOPT time: 3.1543238646000002
Average Python time: 3.451626014709473
Average constraint violation: {'eq_l2': np.float64(1.2911995086471716e-14), 'ineq_max': np.float64(5.1032676174980285), 'ineq_l2': np.float64(5.1032676174980285), 'bound_l2': np.float64(0.0)}
Average KKT residual: 2.6803658273716855